## Streamlined Building Unit Cleaning Code, for San Diego County

Used for joining with SDGE utility

In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona
import pandas as pd

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

In [2]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

In [3]:
# load parcels only for San Diego county (will take about 5 minutes)
parcels = gpd.read_file(
   "/../../capstone/electrigrid/data/parcels/Parcels_CA_2014.gdb",
    layer="CA_PARCELS_STATEWIDE",
    where="County='San Diego'").to_crs(epsg=4326)

In [4]:
# import Zillow data 
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

Building Footprint

Access: https://sat-io.earthengine.app/view/gba

In [5]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

## Streamlined Code (takes 1 minute!)

In [ ]:
## select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi, predicate="contains")

# select the parcels that match these indices
# parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(valid_parcels) < len(parcels)

# drop index right 
valid_parcels = valid_parcels.drop(columns = ['index_right', 'ADDRESS', 'CITY', 'ZIP'])

In [12]:
# contains the valid parcels and the zillow data
valid_parcels

,PARNO,County,Shape_Length,Shape_Area,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code
23,4241720700,San Diego,98.936758,434.984710,"MULTIPOLYGON Z (((-117.23498 32.79751 0.00000,...",Multi,1984.0,6.0,None,None,None,3.0,853387.0,living,2793.0,7997595,06073007908,h530,SDGE,RI110
118,4241721300,San Diego,76.116160,347.815384,"MULTIPOLYGON Z (((-117.23439 32.79723 0.00000,...",Multi,1941.0,2.0,None,None,O,2.0,464641.0,living,1061.0,7997601,06073007908,h530,SDGE,RI101
119,4241722000,San Diego,106.557535,579.922629,"MULTIPOLYGON Z (((-117.23549 32.79699 0.00000,...",Multi,1940.0,3.0,None,None,None,3.0,803219.0,living,1582.0,7997608,06073007908,h530,SDGE,RI110
151,4236921300,San Diego,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101
154,4236220200,San Diego,67.240870,224.046738,"MULTIPOLYGON Z (((-117.25236 32.77776 0.00000,...",Multi,1950.0,3.0,None,None,None,3.0,845088.0,living,1413.0,7995753,06073007601,h530,SDGE,RI110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1039997,2581831500,San Diego,91.675357,462.777931,"MULTIPOLYGON Z (((-117.29489 33.04032 0.00000,...",Multi,2000.0,NaN,None,None,None,2.0,136948.0,living,1811.0,7726906,06073017501,h635,SDGE,RI101
1040095,5182013023,San Diego,784.819873,13648.161643,"MULTIPOLYGON Z (((-116.92353 32.75087 0.00000,...",Multi,1986.0,3.0,None,None,O,1.0,520014.0,living,1294.0,8157714,06073013604,h366,SDGE,RI101
1040096,5182013025,San Diego,784.819873,13648.161643,"MULTIPOLYGON Z (((-116.92353 32.75087 0.00000,...",Multi,1986.0,3.0,None,None,O,1.0,520014.0,living,1294.0,8157714,06073013604,h366,SDGE,RI101
1040121,2181606400,San Diego,152.632191,1299.475183,"MULTIPOLYGON Z (((-117.15515 33.15440 0.00000,...",Multi,1976.0,6.0,None,None,None,2.0,290793.0,living,2048.0,7653376,06073020041,h495,SDGE,RI101


In [13]:

# crop to multi-family residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(valid_parcels, how = "right", predicate="intersects")
# buildings_res = building.loc[valid_buildings]

## join parcels to buildings (keeping observations as parcels, but with building attributes)
# sum number of units per parcel
# units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# # join on parcels with summed number of units
# parcels_res = parcels_res.join(units_sum)

# confirm that joining with Zillow decreased the number of buildings
assert len(valid_buildings) < len(building)

In [14]:
# contains buildings, parcels, and zillow data
valid_buildings

,index_left,source,id,height,var,region,bbox,PARNO,County,Shape_Length,Shape_Area,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code
23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4241720700,San Diego,98.936758,434.984710,"MULTIPOLYGON Z (((-117.23498 32.79751 0.00000,...",Multi,1984.0,6.0,None,None,None,3.0,853387.0,living,2793.0,7997595,06073007908,h530,SDGE,RI110
118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4241721300,San Diego,76.116160,347.815384,"MULTIPOLYGON Z (((-117.23439 32.79723 0.00000,...",Multi,1941.0,2.0,None,None,O,2.0,464641.0,living,1061.0,7997601,06073007908,h530,SDGE,RI101
119,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4241722000,San Diego,106.557535,579.922629,"MULTIPOLYGON Z (((-117.23549 32.79699 0.00000,...",Multi,1940.0,3.0,None,None,None,3.0,803219.0,living,1582.0,7997608,06073007908,h530,SDGE,RI110
151,7349118.0,osm,1308122119,3.827157,0.188816,USA,"{'xmax': -117.25130939999998, 'xmin': -117.251...",4236921300,San Diego,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101
151,7349122.0,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",4236921300,San Diego,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1040096,7521538.0,osm,1120093375,1.555912,0.439441,USA,"{'xmax': -116.92366499999999, 'xmin': -116.923...",5182013025,San Diego,784.819873,13648.161643,"MULTIPOLYGON Z (((-116.92353 32.75087 0.00000,...",Multi,1986.0,3.0,None,None,O,1.0,520014.0,living,1294.0,8157714,06073013604,h366,SDGE,RI101
1040096,7521549.0,osm,1085536694,8.260902,0.931419,USA,"{'xmax': -116.92442420000002, 'xmin': -116.924...",5182013025,San Diego,784.819873,13648.161643,"MULTIPOLYGON Z (((-116.92353 32.75087 0.00000,...",Multi,1986.0,3.0,None,None,O,1.0,520014.0,living,1294.0,8157714,06073013604,h366,SDGE,RI101
1040121,6607446.0,ms,UnitedStates_023013203_67117,8.271330,2.361229,USA,"{'xmax': -117.1550661828501, 'xmin': -117.1553...",2181606400,San Diego,152.632191,1299.475183,"MULTIPOLYGON Z (((-117.15515 33.15440 0.00000,...",Multi,1976.0,6.0,None,None,None,2.0,290793.0,living,2048.0,7653376,06073020041,h495,SDGE,RI101
1040121,6607448.0,ms,UnitedStates_023013203_244602,3.315145,0.346924,USA,"{'xmax': -117.15498855077742, 'xmin': -117.155...",2181606400,San Diego,152.632191,1299.475183,"MULTIPOLYGON Z (((-117.15515 33.15440 0.00000,...",Multi,1976.0,6.0,None,None,None,2.0,290793.0,living,2048.0,7653376,06073020041,h495,SDGE,RI101


In [10]:
# there are some parcels with no buildings (?)
valid_buildings['height'].isna().sum()

1949

In [15]:

# keep all residential buildings, and add zillow points only where they match up
# building_zillow = gpd.sjoin(
#     buildings_res,
#     zillow_multi,
#     how = "left",
#     predicate = "intersects")

# reproject data frame to crs with meters as units
building_m = valid_buildings.to_crs("EPSG:6933")

# drop the index_right column
# building_m = building_m.drop(columns = 'index_right')

KeyboardInterrupt: 

In [32]:
building_m

,source,id,height,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code
6344544,osm,546629542,2.095237,0.773937,USA,"{'xmax': -116.79879449999999, 'xmin': -116.799...","POLYGON ((-11269492.035 4028183.463, -11269502...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6344637,osm,546630246,2.132655,0.123639,USA,"{'xmax': -116.7952663, 'xmin': -116.7954647999...","POLYGON ((-11269152.288 4028086.086, -11269159...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6344638,ms,UnitedStates_023013203_218928,8.093865,2.164009,USA,"{'xmax': -116.79542502150233, 'xmin': -116.795...","POLYGON ((-11269168.007 4028094.660, -11269156...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6344639,ms,UnitedStates_023013203_277953,6.538584,2.185169,USA,"{'xmax': -116.7951542354982, 'xmin': -116.7952...","POLYGON ((-11269142.674 4028019.099, -11269137...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6344640,ms,UnitedStates_023013203_277952,6.395244,3.757385,USA,"{'xmax': -116.79521751197618, 'xmin': -116.795...","POLYGON ((-11269136.455 4027943.939, -11269136...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7720646,ms,UnitedStates_023013221_617530,3.968409,0.611403,USA,"{'xmax': -116.76287700700234, 'xmin': -116.763...","POLYGON ((-11266032.966 3945493.403, -11266032...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7720648,ms,UnitedStates_023013221_20839,4.138736,1.040947,USA,"{'xmax': -116.76272701637753, 'xmin': -116.762...","POLYGON ((-11266009.611 3945444.311, -11266009...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7720672,ms,UnitedStates_023013221_415526,1.914224,0.045000,USA,"{'xmax': -116.7523910988333, 'xmin': -116.7525...","POLYGON ((-11265017.696 3945039.758, -11265003...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7720673,ms,UnitedStates_023013221_118474,1.503267,0.086076,USA,"{'xmax': -116.75161930609684, 'xmin': -116.751...","POLYGON ((-11264929.460 3944958.166, -11264935...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
building_m['unit'].unique()

array([ nan,   2.,   3.,   4.,   6.,   1.,  36.,  68.,   5.,  14.,  13.,
        28., 138., 218.,  16.,  11.,  32.,  52.,  62.,  70., 330., 119.,
        24., 178., 402.,   7.,  10.,  83.,  80.,   8.,   9.,  27.,  17.,
       112.,  29.,  82.,  22., 204., 140., 114.,  69., 152.,  96.,  44.,
        66., 146.,  20.,  12.,  56.,  19.,  30.,  47.,  26.,  58.,  31.,
       288., 157.,  74.,  78.,  50.,  21.,  43.,  64., 278., 106., 300.,
       132., 143.,  60.,  51.,  45.,  15.,  18.,  41.,  57.,  25.,  72.,
        92., 104., 108., 186., 136.,  37.,  90.,  40.,  97.,  35., 180.,
        46., 160.,  23., 156., 220., 240.,  42.,  48.,  33., 336., 168.,
       154.,  76., 179.,  34.,  81.,  38., 264., 148., 184., 340., 120.,
       372., 149., 208.,  93., 122.,  63., 192., 196., 100.,  61.,  49.,
       109., 102., 165.,  75., 117., 169.,  91., 130., 314., 183.,  86.,
        95., 101.,  73.,  98., 212., 150., 328., 216., 127., 226.,  88.,
        84., 116., 202.,  55.,  59.,  65., 126.,  5

In [35]:
# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

In [36]:
building_m

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3
6344544,osm,546629542,2.095237,0.773937,USA,"{'xmax': -116.79879449999999, 'xmin': -116.799...","POLYGON ((-11269492.035 4028183.463, -11269502...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,241.715936,506.452119
6344637,osm,546630246,2.132655,0.123639,USA,"{'xmax': -116.7952663, 'xmin': -116.7954647999...","POLYGON ((-11269152.288 4028086.086, -11269159...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,147.837490,315.286347
6344638,ms,UnitedStates_023013203_218928,8.093865,2.164009,USA,"{'xmax': -116.79542502150233, 'xmin': -116.795...","POLYGON ((-11269168.007 4028094.660, -11269156...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,199.453610,1614.350670
6344639,ms,UnitedStates_023013203_277953,6.538584,2.185169,USA,"{'xmax': -116.7951542354982, 'xmin': -116.7952...","POLYGON ((-11269142.674 4028019.099, -11269137...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,99.570669,651.051204
6344640,ms,UnitedStates_023013203_277952,6.395244,3.757385,USA,"{'xmax': -116.79521751197618, 'xmin': -116.795...","POLYGON ((-11269136.455 4027943.939, -11269136...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,123.341562,788.799342
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7720646,ms,UnitedStates_023013221_617530,3.968409,0.611403,USA,"{'xmax': -116.76287700700234, 'xmin': -116.763...","POLYGON ((-11266032.966 3945493.403, -11266032...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,263.339151,1045.037473
7720648,ms,UnitedStates_023013221_20839,4.138736,1.040947,USA,"{'xmax': -116.76272701637753, 'xmin': -116.762...","POLYGON ((-11266009.611 3945444.311, -11266009...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.401541,452.784123
7720672,ms,UnitedStates_023013221_415526,1.914224,0.045000,USA,"{'xmax': -116.7523910988333, 'xmin': -116.7525...","POLYGON ((-11265017.696 3945039.758, -11265003...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93.955237,179.851361
7720673,ms,UnitedStates_023013221_118474,1.503267,0.086076,USA,"{'xmax': -116.75161930609684, 'xmin': -116.751...","POLYGON ((-11264929.460 3944958.166, -11264935...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,106.674333,160.360009


In [37]:
# keep only observations with unit data
building_w_units = building_m[~building_m['unit'].isna()]

assert building_w_units['unit'].isna().sum() == 0

# run model
results = smf.ols('unit ~ volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

/Users/sofiarodas/.conda/envs/electrigrid-env/lib/python3.11/site-packages/geopandas/geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [38]:
building_w_units

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3,residual
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-11288598.440 4018349.084, -11288592...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446.0,06073019110,h221,SDGE,RI110,412.421351,1606.752247,-5.151029
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-11287597.377 4013374.612, -11287563...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631.0,06073019110,h464,SDGE,RI110,452.260448,1858.907171,-5.575874
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665.0,06073021002,h479,SDGE,RI101,232.343850,905.530915,-3.969574
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-11228325.006 4013391.447, -11228324...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624.0,06073021002,h479,SDGE,RI101,284.844083,1177.213436,-4.427319
6353689,osm,1136229915,3.878115,0.254450,USA,"{'xmax': -116.37282159999998, 'xmin': -116.372...","POLYGON ((-11228395.315 4013393.814, -11228381...",Multi,1962.0,4.0,None,None,None,2.0,155077.0,living,1736.0,7599626.0,06073021002,h479,SDGE,RI101,342.459460,1328.097228,-4.681536
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7714237,osm,1120169796,7.944537,3.131626,USA,"{'xmax': -116.98553089999999, 'xmin': -116.985...","POLYGON ((-11287532.828 3945273.039, -11287522...",Multi,2013.0,NaN,None,None,None,278.0,45390902.0,None,NaN,8336746.0,06073013317,h493,SDGE,RI104,580.892082,4614.918737,265.780649
7715428,osm,1120169118,9.802270,3.607025,USA,"{'xmax': -116.98862620000001, 'xmin': -116.989...","POLYGON ((-11287835.939 3944949.530, -11287830...",Multi,2015.0,NaN,None,None,None,187.0,50592225.0,None,NaN,8400963.0,06073013317,h493,SDGE,RI104,903.386384,8855.237190,167.636331
7715815,osm,1120565410,4.715566,0.911623,USA,"{'xmax': -116.97998779999999, 'xmin': -116.980...","POLYGON ((-11286978.360 3944699.625, -11286979...",Multi,2010.0,5.0,None,None,O,2.0,751590.0,living,2992.0,8336605.0,06073013317,h493,SDGE,RI101,196.444060,926.344867,-4.004642
7717377,osm,1009026928,10.977699,6.832924,USA,"{'xmax': -116.96362569999997, 'xmin': -116.964...","POLYGON ((-11285483.179 3945189.515, -11285398...",Multi,2018.0,NaN,None,None,None,123.0,10388986.0,living,73407.0,8414651.0,06073013319,h493,SDGE,RI104,3435.935102,37718.662300,55.005671


In [39]:
# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

In [40]:
building_units_clean.shape

(22889, 25)

In [41]:
building_outliers.shape

(450, 25)

In [42]:
# rerun linear regression
results_clean = smf.ols('unit ~ volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

/tmp/ipykernel_1869795/3412973521.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept = results_clean.params[0]
/tmp/ipykernel_1869795/3412973521.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = results_clean.params[1]


In [43]:
intercept

2.1071031942453295

In [44]:
slope

0.0018849863132773506

In [45]:
# extract just the multi-family homes where unit info is missing
missing_units = building_m[building_m['unit'].isna()]

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units], axis = 0)

assert len(missing_units) < len(missing_outlier_units)

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['unit'] = round(intercept + missing_outlier_units_pred['volume_m3'] * slope)

In [46]:
missing_outlier_units_pred['unit'].agg(['min','max'])

min      2.0
max    772.0
Name: unit, dtype: float64

In [47]:
# combine multi-family homes data frames
multi_complete = pd.concat([building_w_units, missing_outlier_units_pred]).to_crs(zillow.crs)

# drop residual column
multi_complete = multi_complete.drop(['residual'], axis = 1)


In [48]:
multi_complete

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-116.99693 33.29913, -116.99687 33.2...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446.0,06073019110,h221,SDGE,RI110,412.421351,1606.752247
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-116.98655 33.25268, -116.98620 33.2...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631.0,06073019110,h464,SDGE,RI110,452.260448,1858.907171
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-116.37161 33.25203, -116.37152 33.2...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665.0,06073021002,h479,SDGE,RI101,232.343850,905.530915
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-116.37224 33.25284, -116.37224 33.2...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624.0,06073021002,h479,SDGE,RI101,284.844083,1177.213436
6353689,osm,1136229915,3.878115,0.254450,USA,"{'xmax': -116.37282159999998, 'xmin': -116.372...","POLYGON ((-116.37297 33.25286, -116.37283 33.2...",Multi,1962.0,4.0,None,None,None,2.0,155077.0,living,1736.0,7599626.0,06073021002,h479,SDGE,RI101,342.459460,1328.097228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101350,ms,UnitedStates_023013221_617530,3.968409,0.611403,USA,"{'xmax': -116.76287700700234, 'xmin': -116.763...","POLYGON ((-116.76306 32.62119, -116.76305 32.6...",NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,263.339151,1045.037473
101351,ms,UnitedStates_023013221_20839,4.138736,1.040947,USA,"{'xmax': -116.76272701637753, 'xmin': -116.762...","POLYGON ((-116.76281 32.62073, -116.76281 32.6...",NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.401541,452.784123
101352,ms,UnitedStates_023013221_415526,1.914224,0.045000,USA,"{'xmax': -116.7523910988333, 'xmin': -116.7525...","POLYGON ((-116.75253 32.61698, -116.75239 32.6...",NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,93.955237,179.851361
101353,ms,UnitedStates_023013221_118474,1.503267,0.086076,USA,"{'xmax': -116.75161930609684, 'xmin': -116.751...","POLYGON ((-116.75162 32.61623, -116.75169 32.6...",NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,106.674333,160.360009


In [ ]:
# check that the unit data is complete
multi_complete['unit'].isna().any()

False

In [ ]:

# WHY DO WE DO THIS AGAIN
# multi_by_parcel = parcels_res.sjoin(multi_complete, predicate="intersects")

# the two unit columns don't match this makes sense since the one is outliers and projected
# multi_by_parcel['unit_left'].equals(multi_by_parcel['unit_right'])
## IF WE DO NEED THIS WE WOULD KEEP UNIT RIGHT AS THATS THE ONE THATS COMING FROM UNIT COMPLETE

# IF WE DO NEED PARNO CHECK THE DUPLICATES
# check how many parcel numbers are duplicates
# duplicates = multi_complete.pivot_table(index = ['PARNO'], aggfunc = 'size')
# print(f"There are ",duplicates[duplicates > 1].sum(), " duplicated parcels.")


In [40]:
multi_by_parcel = multi_by_parcel.to_crs("EPSG:6933")

In [ ]:
## IN ORDER TO CONCAT THE MULTI-FAMILY HOMES WITH SINGLE FAMILY HOMES THEY MUST ALL BE POINTS
# create centroids for each multi residential parcel 
multi_by_parcel['centroids'] = multi_by_parcel.geometry.centroid


In [42]:
multi_by_parcel

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,unit_left,index_right,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,unit_right,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3,centroids
151,4236921300,San Diego,None,None,None,69.352091,238.641330,MULTIPOLYGON Z (((-11313159.182 3961251.281 0....,2.0,7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,140.235708,431.200368,POINT (-11313153.723 3961239.229)
154,4236220200,San Diego,None,None,None,67.240870,224.046738,MULTIPOLYGON Z (((-11313244.013 3962367.862 0....,3.0,7349569,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,2.0,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,117.733224,320.096607,POINT (-11313238.717 3962356.234)
159,4245420100,San Diego,None,None,None,94.646744,497.462992,MULTIPOLYGON Z (((-11311377.586 3963837.438 0....,4.0,7316916,ms,UnitedStates_023013221_606679,7.757454,4.108285,USA,"{'xmax': -117.23299227742604, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,160936.0,None,NaN,7998883,06073007702,h530,SDGE,RI110,217.808548,1689.639778,POINT (-11311395.134 3963841.243)
159,4245420100,San Diego,None,None,None,94.646744,497.462992,MULTIPOLYGON Z (((-11311377.586 3963837.438 0....,4.0,7316915,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,224.418227,1755.586108,POINT (-11311395.134 3963841.243)
160,4245420700,San Diego,None,None,None,92.385658,398.294157,MULTIPOLYGON Z (((-11311356.480 3963757.406 0....,2.0,7316886,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,2.0,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,113.884017,734.439912,POINT (-11311373.489 3963745.624)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1039825,2632430800,San Diego,None,None,None,137.363218,918.420109,MULTIPOLYGON Z (((-11314226.300 3985891.446 0....,2.0,7367690,ms,UnitedStates_023013221_216529,3.732555,0.827698,USA,"{'xmax': -117.26281072873077, 'xmin': -117.263...",Multi,1969.0,4.0,None,None,O,2.0,1319921.0,living,3047.0,7736193,06073017303,h510,SDGE,RI101,301.800899,1126.488574,POINT (-11314255.652 3985894.577)
1039961,5032808000,San Diego,None,None,None,114.021685,694.906225,MULTIPOLYGON Z (((-11290124.510 3957832.406 0....,2.0,7218469,ms,UnitedStates_023013221_556001,4.677042,0.605590,USA,"{'xmax': -117.01293428361708, 'xmin': -117.013...",Multi,1993.0,6.0,None,None,None,2.0,252491.0,living,2568.0,8141863,06073013802,h302,SDGE,RI110,239.438899,1119.865903,POINT (-11290144.704 3957823.677)
1040095,5182013023,San Diego,None,None,None,784.819873,13648.161643,MULTIPOLYGON Z (((-11281516.606 3959472.038 0....,1.0,7521513,osm,1085544737,7.041645,4.969846,USA,"{'xmax': -116.92388439999999, 'xmin': -116.924...",Multi,1986.0,3.0,None,None,O,1.0,520014.0,living,1294.0,8157714,06073013604,h366,SDGE,RI101,237.086541,1669.479270,POINT (-11281579.243 3959538.970)
1040096,5182013025,San Diego,None,None,None,784.819873,13648.161643,MULTIPOLYGON Z (((-11281516.606 3959472.038 0....,1.0,7521513,osm,1085544737,7.041645,4.969846,USA,"{'xmax': -116.92388439999999, 'xmin': -116.924...",Multi,1986.0,3.0,None,None,O,1.0,520014.0,living,1294.0,8157714,06073013604,h366,SDGE,RI101,237.086541,1669.479270,POINT (-11281579.243 3959538.970)


In [43]:

# drop the geometry of multi_by_parcel so the centroid becomes the geometry
multi_by_parcel = multi_by_parcel.drop(columns = 'geometry')

# set the centroid to the geometry
multi_by_parcel_processed = multi_by_parcel.set_geometry('centroids')


In [63]:
multi_by_parcel_processed

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,unit_left,index_right,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,unit_right,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3,0
151,4236921300,San Diego,None,None,None,69.352091,238.641330,2.0,7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,140.235708,431.200368,POINT (-11313153.723 3961239.229)
154,4236220200,San Diego,None,None,None,67.240870,224.046738,3.0,7349569,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,2.0,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,117.733224,320.096607,POINT (-11313238.717 3962356.234)
159,4245420100,San Diego,None,None,None,94.646744,497.462992,4.0,7316916,ms,UnitedStates_023013221_606679,7.757454,4.108285,USA,"{'xmax': -117.23299227742604, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,160936.0,None,NaN,7998883,06073007702,h530,SDGE,RI110,217.808548,1689.639778,POINT (-11311395.134 3963841.243)
159,4245420100,San Diego,None,None,None,94.646744,497.462992,4.0,7316916,ms,UnitedStates_023013221_606679,7.757454,4.108285,USA,"{'xmax': -117.23299227742604, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,160936.0,None,NaN,7998883,06073007702,h530,SDGE,RI110,217.808548,1689.639778,POINT (-11311395.134 3963841.243)
159,4245420100,San Diego,None,None,None,94.646744,497.462992,4.0,7316915,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,224.418227,1755.586108,POINT (-11311395.134 3963841.243)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1039825,2632430800,San Diego,None,None,None,137.363218,918.420109,2.0,7367690,ms,UnitedStates_023013221_216529,3.732555,0.827698,USA,"{'xmax': -117.26281072873077, 'xmin': -117.263...",Multi,1969.0,4.0,None,None,O,2.0,1319921.0,living,3047.0,7736193,06073017303,h510,SDGE,RI101,301.800899,1126.488574,POINT (-11314255.652 3985894.577)
1039961,5032808000,San Diego,None,None,None,114.021685,694.906225,2.0,7218469,ms,UnitedStates_023013221_556001,4.677042,0.605590,USA,"{'xmax': -117.01293428361708, 'xmin': -117.013...",Multi,1993.0,6.0,None,None,None,2.0,252491.0,living,2568.0,8141863,06073013802,h302,SDGE,RI110,239.438899,1119.865903,POINT (-11290144.704 3957823.677)
1040095,5182013023,San Diego,None,None,None,784.819873,13648.161643,1.0,7521513,osm,1085544737,7.041645,4.969846,USA,"{'xmax': -116.92388439999999, 'xmin': -116.924...",Multi,1986.0,3.0,None,None,O,1.0,520014.0,living,1294.0,8157714,06073013604,h366,SDGE,RI101,237.086541,1669.479270,POINT (-11281579.243 3959538.970)
1040096,5182013025,San Diego,None,None,None,784.819873,13648.161643,1.0,7521513,osm,1085544737,7.041645,4.969846,USA,"{'xmax': -116.92388439999999, 'xmin': -116.924...",Multi,1986.0,3.0,None,None,O,1.0,520014.0,living,1294.0,8157714,06073013604,h366,SDGE,RI101,237.086541,1669.479270,POINT (-11281579.243 3959538.970)


**NOT SURE WHERE THE INDEX RIGHT COMES FROM**

In [44]:
# change the CRS back to the Zillow CRS 
multi_by_parcel_processed = multi_by_parcel_processed.to_crs(zillow.crs)

# subset for single family zillow and condos
non_multi_zillow = zillow[zillow['type'] != "Multi"].to_crs(zillow.crs)

# concat the single and multifamily dataframes
complete_zillow = pd.concat([multi_by_parcel_processed, non_multi_zillow], axis = 0)

In [45]:
complete_zillow

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,unit_left,index_right,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,unit_right,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3,centroids,unit,geometry
151,4236921300,San Diego,None,None,None,69.352091,238.641330,2.0,7349122.0,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,140.235708,431.200368,POINT (-117.25142 32.76728),NaN,None
154,4236220200,San Diego,None,None,None,67.240870,224.046738,3.0,7349569.0,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,2.0,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,117.733224,320.096607,POINT (-117.25230 32.77765),NaN,None
159,4245420100,San Diego,None,None,None,94.646744,497.462992,4.0,7316916.0,ms,UnitedStates_023013221_606679,7.757454,4.108285,USA,"{'xmax': -117.23299227742604, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,160936.0,None,NaN,7998883,06073007702,h530,SDGE,RI110,217.808548,1689.639778,POINT (-117.23320 32.79144),NaN,None
159,4245420100,San Diego,None,None,None,94.646744,497.462992,4.0,7316915.0,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,224.418227,1755.586108,POINT (-117.23320 32.79144),NaN,None
160,4245420700,San Diego,None,None,None,92.385658,398.294157,2.0,7316886.0,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,2.0,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,113.884017,734.439912,POINT (-117.23297 32.79055),NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10012615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,2.0,None,None,O,NaN,1737925.0,living,2408.0,10714295,06111002500,h2540,Others,RR101,NaN,NaN,None,NaN,POINT (-119.26588 34.25417)
10012616,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,4.0,None,None,O,NaN,921196.0,living,3301.0,10714296,06111002500,h2540,Others,RR101,NaN,NaN,None,NaN,POINT (-119.26591 34.25429)
10012617,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,4.0,None,None,O,NaN,1503087.0,living,2667.0,10714297,06111002500,h2540,Others,RR101,NaN,NaN,None,NaN,POINT (-119.26594 34.25441)
10012618,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,4.0,None,None,O,NaN,998162.0,living,3011.0,10714430,06111002500,h2540,Others,RR101,NaN,NaN,None,NaN,POINT (-119.26252 34.25562)


In [ ]:


# # join back to parcels
# units_multi = parcels_res.sjoin(multi_complete, predicate="intersects").groupby(level=0)['unit_right'].sum()

# # join on parcels with summed number of units
# multi_summed_units = parcels_res.join(units_multi)

# assert len(multi_summed_units) < len(multi_by_parcel)

# # save all non-multi observations
# non_multi = building_m[building_m['type'] != "Multi"].to_crs(zillow.crs)


# keep only variables of interest
# non_multi = non_multi[['source', 'id', 'height_m', 'var', 'region', 'bbox', 'geometry', 'area_m2', 'volume_m3']]

# join Zillow data to non-multi family homes (takes ~1 minute)
# non_multi_points = gpd.sjoin(
#     zillow, # left df's geometry is always kept
#     non_multi,
#     how = "inner",
#     predicate = "intersects")


## Final results

In [38]:
multi_complete.head() # should be my buildings??

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-116.99693 33.29913, -116.99687 33.2...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,412.421351,1606.752247
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-116.98655 33.25268, -116.98620 33.2...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,452.260448,1858.907171
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-116.37161 33.25203, -116.37152 33.2...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,232.343850,905.530915
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-116.37224 33.25284, -116.37224 33.2...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,284.844083,1177.213436
6353689,osm,1136229915,3.878115,0.254450,USA,"{'xmax': -116.37282159999998, 'xmin': -116.372...","POLYGON ((-116.37297 33.25286, -116.37283 33.2...",Multi,1962.0,4.0,None,None,None,2.0,155077.0,living,1736.0,7599626,06073021002,h479,SDGE,RI101,342.459460,1328.097228


In [8]:
multi_summed_units.head(3)

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,unit,unit_right
23,4241720700,San Diego,None,None,None,98.936758,434.984710,"MULTIPOLYGON Z (((-117.23498 32.79751 0.00000,...",3.0,NaN
118,4241721300,San Diego,None,None,None,76.116160,347.815384,"MULTIPOLYGON Z (((-117.23439 32.79723 0.00000,...",2.0,NaN
119,4241722000,San Diego,None,None,None,106.557535,579.922629,"MULTIPOLYGON Z (((-117.23549 32.79698 0.00000,...",3.0,NaN


In [9]:
multi_by_parcel.head(3)

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,unit_left,index_right,...,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3
151,4236921300,San Diego,None,None,None,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0,80603,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,130.144663,498.084000
151,4236921300,San Diego,None,None,None,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0,7349122,...,1026707.0,None,NaN,7996232.0,06073007602,h567,SDGE,RI101,140.235708,431.200368
154,4236220200,San Diego,None,None,None,67.240870,224.046738,"MULTIPOLYGON Z (((-117.25236 32.77776 0.00000,...",3.0,80751,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.155171,171.388014


In [10]:
non_multi_points.head(3)

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,...,geometry,index_right,source,id,height_m,var,region,bbox,area_m2,volume_m3
7160203,Single,1959.0,3.0,None,None,None,1.0,179462.0,living,1780.0,...,POINT (-117.24993 33.38692),6701574,ms,UnitedStates_023013203_33032,3.835961,0.299546,USA,"{'xmin': -117.25014289023119, 'ymin': 33.38687...",301.592312,1156.896305
7160213,Single,1952.0,2.0,None,None,I,1.0,153443.0,living,1510.0,...,POINT (-117.24838 33.38564),6698354,ms,UnitedStates_023013203_96432,5.221156,0.925386,USA,"{'xmin': -117.24844943872387, 'ymin': 33.38562...",192.020400,1002.568394
7160303,Single,1962.0,3.0,None,None,I,2.0,401364.0,living,1540.0,...,POINT (-117.24780 33.38609),6698329,ms,UnitedStates_023013203_294280,6.511510,1.177574,USA,"{'xmin': -117.24791472273105, 'ymin': 33.38604...",237.514781,1546.579963


## Renaming and saving

In [13]:
multi_summed_units_sd = multi_summed_units.copy()

multi_by_parcel_sd = multi_by_parcel.copy()

non_multi_points_sd = non_multi_points.copy()

In [17]:
# takes a really long time :,(
multi_by_parcel_sd.to_file("data/multi_by_parcel_sd.geojson", driver='GeoJSON')

KeyboardInterrupt: 

In [15]:
non_multi_points_sd.to_file("data/non_multi_zillow_sd.geojson", driver='GeoJSON')

In [16]:
multi_summed_units_sd.to_file("data/multi_summed_units_sd.geojson", driver='GeoJSON')

In [14]:
## Saving! (takes at least 40 minutes)

multi_summed_units_sd.to_file("data/multi_summed_units_sd.geojson", driver='GeoJSON')

multi_by_parcel_sd.to_file("data/multi_by_parcel_sd.geojson", driver='GeoJSON')

non_multi_points_sd.to_file("data/non_multi_zillow_sd.geojson", driver='GeoJSON')

KeyboardInterrupt: 